# Multi-seed ablation with dispersion, corrected tests, and equivalence tests

The ablation grid reported so far is a single seed with no dispersion, and the paper's own
seed-stability measurement puts the respiratory AUC range at $0.701$–$0.711$ across five
seeds. Most of the interpreted ablation differences are smaller than that. This notebook
therefore re-runs every condition across three seeds and asks, for each one, whether the
difference survives its own noise.

Three things are computed that were previously absent:

1. **Dispersion.** Per-fold mean ± SD for every condition and every metric.
2. **Corrected significance.** Paired Wilcoxon across folds for *every* comparison against
   the full model, with Holm correction — not one test out of sixteen.
3. **Equivalence, not just absence of significance.** Where the claim is that a modality does
   *not* matter, a non-significant p-value is not evidence of no effect. Two one-sided tests
   (TOST) against a pre-stated margin are reported instead. The margin is set to one
   fold standard deviation of the full model, declared before the runs.

The cumulative build-up is included in the same sweep, because the claim that SpO$_2$ alone
reaches the full AUC is load-bearing for the paper's non-circularity argument and is
currently reported in a single clause with no numbers behind it.

In [1]:
import itertools
import json
import os
import sys
import time

import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
os.makedirs(OUT, exist_ok=True)
CACHE = os.path.join(OUT, "multiseed_ablation.json")
SEEDS = [42, 1, 7]

CARD_ALL = ["spo2", "pulse_hrv", "ecg", "airflow", "effort"]

# name -> (fusion, eeg_drop, card_drop)
CONDITIONS = {
    "full":              ("concat", (), ()),
    "-SpO2":             ("concat", (), ("spo2",)),
    "-effort":           ("concat", (), ("effort",)),
    "-pulse/HRV":        ("concat", (), ("pulse_hrv",)),
    "-ECG":              ("concat", (), ("ecg",)),
    "-airflow":          ("concat", (), ("airflow",)),
    "-EOG":              ("concat", ("eog",), ()),
    "-EMG":              ("concat", ("emg",), ()),
    "-all cardio":       ("concat", (), tuple(CARD_ALL)),
    # cumulative build-up: keep only the named group(s)
    "SpO2 only":         ("concat", (), tuple(g for g in CARD_ALL if g != "spo2")),
    "SpO2+effort":       ("concat", (), tuple(g for g in CARD_ALL if g not in ("spo2", "effort"))),
    "SpO2+effort+flow":  ("concat", (), tuple(g for g in CARD_ALL if g not in ("spo2", "effort", "airflow"))),
    # architecture variant the paper asserts is equivalent but never tabulates
    "attention fusion":  ("cross",  (), ()),
}
print("conditions: %d | seeds: %s | total runs: %d"
      % (len(CONDITIONS), SEEDS, len(CONDITIONS) * len(SEEDS)))

cwd: D:\sleep-staging-psg\MMNet_research\MMNet_Submission\all_codes\notebooks | device: cuda | NVIDIA GeForce RTX 2060


subjects: 96 (SN28 dropped) | epochs: 89,532
stage %: {'W': np.float64(26.6), 'N1': np.float64(10.2), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}
respiratory-event prevalence: 16.0%
EEG-feature counts -> EEG: 112 EOG: 50 EMG: 26
parameters (concat): 773,254
training utilities defined.
conditions: 13 | seeds: [42, 1, 7] | total runs: 39


In [2]:
results = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
t0 = time.time()
todo = [(n, s) for n in CONDITIONS for s in SEEDS if "%s|%d" % (n, s) not in results]
print("%d runs remaining\n" % len(todo))

for name, seed in todo:
    fusion, eeg_drop, card_drop = CONDITIONS[name]
    r = C.run_10fold(fusion=fusion, eeg_drop=eeg_drop, card_drop=card_drop, seed=seed)
    results["%s|%d" % (name, seed)] = {
        "acc":   [f["acc"] for f in r["per_fold"]],
        "mf1":   [f["mf1"] for f in r["per_fold"]],
        "kappa": [f["kappa"] for f in r["per_fold"]],
        "auc":   [f["auc"] for f in r["per_fold"]],
        "ap":    [f["ap"] for f in r["per_fold"]],
    }
    json.dump(results, open(CACHE, "w"))          # checkpoint after every run
    print("%-20s seed %-4d acc %.4f  AUC %.4f   (%.1f min elapsed)"
          % (name, seed, np.mean(results["%s|%d" % (name, seed)]["acc"]),
             np.mean(results["%s|%d" % (name, seed)]["auc"]),
             (time.time() - t0) / 60), flush=True)

print("\ntotal %.1f min" % ((time.time() - t0) / 60))

0 runs remaining


total 0.0 min


## 1. Every condition with its dispersion

Pooling the three seeds, each condition has 30 fold-scores. Reported as mean ± SD across
folds, and separately the spread of the three seed-means, so the reader can see which is
larger.

In [3]:
def collect(name, metric):
    """(all fold values pooled over seeds, list of per-seed means)"""
    per_seed = [results["%s|%d" % (name, s)][metric] for s in SEEDS
                if "%s|%d" % (name, s) in results]
    pooled = [v for run in per_seed for v in run]
    return np.array(pooled), np.array([np.mean(r) for r in per_seed])

print("%-20s %-22s %-22s" % ("", "staging accuracy", "respiratory AUC"))
print("%-20s %-22s %-22s" % ("condition", "mean +- fold SD (seed SD)", "mean +- fold SD (seed SD)"))
print("-" * 66)
for name in CONDITIONS:
    a, a_seed = collect(name, "acc")
    u, u_seed = collect(name, "auc")
    print("%-20s %.4f +-%.3f (%.3f)   %.4f +-%.3f (%.3f)"
          % (name, a.mean(), a.std(), a_seed.std(),
             np.nanmean(u), np.nanstd(u), u_seed.std()))

                     staging accuracy       respiratory AUC       
condition            mean +- fold SD (seed SD) mean +- fold SD (seed SD)
------------------------------------------------------------------
full                 0.7238 +-0.034 (0.003)   0.7050 +-0.033 (0.004)
-SpO2                0.7242 +-0.027 (0.004)   0.6806 +-0.026 (0.004)
-effort              0.7237 +-0.032 (0.004)   0.7207 +-0.040 (0.004)
-pulse/HRV           0.7215 +-0.030 (0.001)   0.7012 +-0.032 (0.001)
-ECG                 0.7229 +-0.031 (0.003)   0.7044 +-0.031 (0.005)
-airflow             0.7232 +-0.029 (0.001)   0.6997 +-0.038 (0.006)
-EOG                 0.7110 +-0.032 (0.001)   0.7028 +-0.032 (0.004)
-EMG                 0.7257 +-0.035 (0.002)   0.7020 +-0.034 (0.004)
-all cardio          0.7265 +-0.027 (0.003)   0.6655 +-0.032 (0.007)
SpO2 only            0.7251 +-0.032 (0.007)   0.7120 +-0.036 (0.001)
SpO2+effort          0.7225 +-0.030 (0.003)   0.6974 +-0.030 (0.005)
SpO2+effort+flow     0.7211 +-0.03

## 2. Does each difference survive its own noise?

Paired Wilcoxon across the 30 pooled folds against the full model, Holm-corrected over all
comparisons within each metric. Alongside it, a TOST equivalence test against a margin of
one full-model fold SD: a significant TOST means the condition is *statistically
indistinguishable* from the full model, which is the claim the paper needs when it says a
modality does not matter.

In [4]:
from scipy.stats import wilcoxon, ttest_rel

def holm(pvals):
    order = np.argsort(pvals)
    m, adj, run = len(pvals), np.empty(len(pvals)), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * pvals[i])
        adj[i] = min(1.0, run)
    return adj

def tost(diff, margin):
    """Two one-sided tests: is |mean difference| inside +-margin?"""
    p_lo = ttest_rel(diff, np.full_like(diff, -margin)).pvalue / 2 if np.mean(diff) > -margin else 1.0
    p_hi = ttest_rel(diff, np.full_like(diff, margin)).pvalue / 2 if np.mean(diff) < margin else 1.0
    return max(p_lo, p_hi)

for metric, label in (("acc", "staging accuracy"), ("auc", "respiratory AUC")):
    base, _ = collect("full", metric)
    margin = float(np.nanstd(base))
    names = [n for n in CONDITIONS if n != "full"]
    raws, deltas, tosts = [], [], []
    for n in names:
        v, _ = collect(n, metric)
        d = v - base
        ok = ~np.isnan(d)
        deltas.append(float(np.mean(d[ok])))
        raws.append(wilcoxon(v[ok], base[ok]).pvalue)
        tosts.append(tost(d[ok], margin))
    adj = holm(np.array(raws))

    print("\n%s   (equivalence margin = one full-model fold SD = %.4f)" % (label.upper(), margin))
    print("%-20s %9s %10s %10s %11s  %s" % ("condition", "delta", "p raw", "p Holm", "p TOST", "verdict"))
    print("-" * 78)
    for n, d, pr, pa, pt in zip(names, deltas, raws, adj, tosts):
        if pa < 0.05:
            verdict = "REAL EFFECT"
        elif pt < 0.05:
            verdict = "equivalent (no effect)"
        else:
            verdict = "INCONCLUSIVE"
        print("%-20s %+9.4f %10.4f %10.4f %11.4f  %s" % (n, d, pr, pa, pt, verdict))


STAGING ACCURACY   (equivalence margin = one full-model fold SD = 0.0340)
condition                delta      p raw     p Holm      p TOST  verdict
------------------------------------------------------------------------------
-SpO2                  +0.0004     0.9032     1.0000      0.0000  equivalent (no effect)
-effort                -0.0001     0.4771     1.0000      0.0000  equivalent (no effect)
-pulse/HRV             -0.0024     0.2449     1.0000      0.0000  equivalent (no effect)
-ECG                   -0.0009     0.5963     1.0000      0.0000  equivalent (no effect)
-airflow               -0.0007     0.6120     1.0000      0.0000  equivalent (no effect)
-EOG                   -0.0128     0.0008     0.0096      0.0000  REAL EFFECT
-EMG                   +0.0019     0.5978     1.0000      0.0000  equivalent (no effect)
-all cardio            +0.0027     0.2845     1.0000      0.0000  equivalent (no effect)
SpO2 only              +0.0012     0.5978     1.0000      0.0000  equiv

In [5]:
summary = {}
for metric in ("acc", "mf1", "kappa", "auc", "ap"):
    summary[metric] = {}
    for name in CONDITIONS:
        v, seed_means = collect(name, metric)
        summary[metric][name] = dict(mean=float(np.nanmean(v)), fold_sd=float(np.nanstd(v)),
                                     seed_sd=float(seed_means.std()), n_folds=int((~np.isnan(v)).sum()))
json.dump(summary, open(os.path.join(OUT, "multiseed_ablation_summary.json"), "w"), indent=1)
print("wrote multiseed_ablation_summary.json")

wrote multiseed_ablation_summary.json


## Reading the result

The verdict column is the point. **REAL EFFECT** means the difference survives correction for
multiple comparisons. **equivalent** means the condition is statistically indistinguishable
from the full model within a pre-stated margin — which is what licenses a claim that a
modality does not contribute. **INCONCLUSIVE** means the experiment cannot tell, and no claim
in either direction is supported by it: neither that the modality matters, nor that it does
not. Any row of the paper's attribution argument that lands on INCONCLUSIVE has to be
withdrawn rather than reworded.